In [1]:
import numpy as np

d = np.load("../data/processed/windows_fd002.npz")
X_tr, y_tr = d["X_tr"], d["y_tr"]
X_val, y_val = d["X_val"], d["y_val"]
X_test, y_test = d["X_test"], d["y_test"]
RUL_CAP = 125
y_test_cap = np.minimum(y_test, RUL_CAP)   # test labels capped like the training labels

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def nasa_score(y_true, y_pred):
    diff = y_pred - y_true
    return float(np.sum(np.where(diff < 0, np.exp(-diff / 13) - 1, np.exp(diff / 10) - 1)))

print(X_tr.shape, X_val.shape, X_test.shape)
print("Sanity check (should be 0.0 and 0.0):", rmse(y_val, y_val), nasa_score(y_val, y_val))
print("Test engines with true RUL above the cap:", int((y_test > RUL_CAP).sum()), "of", len(y_test))

(37174, 30, 28) (9045, 30, 28) (259, 30, 28)
Sanity check (should be 0.0 and 0.0): 0.0 0.0
Test engines with true RUL above the cap: 57 of 259


In [2]:
from sklearn.linear_model import Ridge

Xf_tr = X_tr.reshape(len(X_tr), -1)
Xf_val = X_val.reshape(len(X_val), -1)
Xf_test = X_test.reshape(len(X_test), -1)

def report(name, val_pred, test_pred):
    print(f"{name:<10}{rmse(y_val, val_pred):>9.1f}{rmse(y_test_cap, test_pred):>14.1f}"
          f"{rmse(y_test, test_pred):>14.1f}{nasa_score(y_test, test_pred):>12.0f}")

print(f"{'model':<10}{'val RMSE':>9}{'test (capped)':>14}{'test (true)':>14}{'score':>12}")

mean_val = np.full_like(y_val, y_tr.mean())
mean_test = np.full(len(y_test), y_tr.mean())
report("mean", mean_val, mean_test)

ridge = Ridge(alpha=1.0).fit(Xf_tr, y_tr)
ridge_val = ridge.predict(Xf_val)
ridge_test = ridge.predict(Xf_test)
report("ridge", ridge_val, ridge_test)

model      val RMSE test (capped)   test (true)       score
mean           41.7          43.6          53.8      124601
ridge          17.8          16.9          29.1       13380


C:\Users\anany\turbofan-rul-prediction\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.494498474447937e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


In [3]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.5,
    early_stopping_rounds=30, random_state=42, n_jobs=-1,
)
xgb.fit(Xf_tr, y_tr, eval_set=[(Xf_val, y_val)], verbose=False)

xgb_val = xgb.predict(Xf_val)
xgb_test = xgb.predict(Xf_test)

print("Trees used:", xgb.best_iteration + 1)
print(f"{'model':<10}{'val RMSE':>9}{'test (capped)':>14}{'test (true)':>14}{'score':>12}")
report("xgboost", xgb_val, xgb_test)

Trees used: 500
model      val RMSE test (capped)   test (true)       score
xgboost        15.1          13.8          26.6        8689


In [4]:
import json, os

os.makedirs("../results", exist_ok=True)

def entry(val_pred, test_pred):
    return {
        "val_rmse": rmse(y_val, val_pred),
        "test_rmse_capped": rmse(y_test_cap, test_pred),
        "test_rmse_true": rmse(y_test, test_pred),
        "test_score": nasa_score(y_test, test_pred),
    }

results = {
    "mean": entry(mean_val, mean_test),
    "ridge": entry(ridge_val, ridge_test),
    "xgboost": entry(xgb_val, xgb_test),
}
with open("../results/baselines_fd002.json", "w") as f:
    json.dump(results, f, indent=2)

print(open("../results/baselines_fd002.json").read())

{
  "mean": {
    "val_rmse": 41.72836685180664,
    "test_rmse_capped": 43.55384063720703,
    "test_rmse_true": 53.779178619384766,
    "test_score": 124600.765625
  },
  "ridge": {
    "val_rmse": 17.833820343017578,
    "test_rmse_capped": 16.877216339111328,
    "test_rmse_true": 29.092309951782227,
    "test_score": 13379.6484375
  },
  "xgboost": {
    "val_rmse": 15.063028335571289,
    "test_rmse_capped": 13.784896850585938,
    "test_rmse_true": 26.648603439331055,
    "test_score": 8688.7412109375
  }
}
